In [1]:
import os
os.chdir("..")      # move up to the project folder
print(os.getcwd())  # should end in gdsc-drug-response-prediction

/Users/ashirenarika


In [2]:
import os
os.chdir(os.path.expanduser("~/gdsc-drug-response-prediction"))
print(os.getcwd())
print(os.listdir("data/raw"))

/Users/ashirenarika/gdsc-drug-response-prediction
['expression_log2tpm_top5000.csv.gz', 'model_info.csv', 'GDSC2_fitted_dose_response_24Jul22.csv', 'screened_compounds_rel_8.4.csv', 'Cell_Lines_Details.xlsx']


In [3]:
import pandas as pd, numpy as np

expr = pd.read_csv("data/raw/expression_log2tpm_top5000.csv.gz", index_col="SANGER_MODEL_ID")
models = pd.read_csv("data/raw/model_info.csv").set_index("SANGER_MODEL_ID")
print(expr.shape)
expr.iloc[:3, :5]

(1898, 5000)


,RPS4Y1,TMSB4XP6,FXYD3,C19orf33,TACSTD2
SANGER_MODEL_ID,,,,,
SIDM00001,3.2325,3.1368,0.7697,0.8733,1.3341
SIDM00002,3.4175,0.0000,0.0407,0.0982,0.1344
SIDM00003,1.9169,0.0000,3.2183,0.6285,0.0407


In [4]:
resp = pd.read_csv("data/raw/GDSC2_fitted_dose_response_24Jul22.csv")

resp = (resp.groupby(["SANGER_MODEL_ID", "DRUG_NAME"], as_index=False)
            .agg(LN_IC50=("LN_IC50", "mean"), AUC=("AUC", "mean"),
                 TISSUE=("TCGA_DESC", "first"), TARGET=("PUTATIVE_TARGET", "first"),
                 PATHWAY=("PATHWAY_NAME", "first")))

print(resp.shape)
print("Share of screened cell lines with expression data:",
      round(resp.SANGER_MODEL_ID.isin(expr.index).mean(), 2))

(235748, 7)
Share of screened cell lines with expression data: 0.97


In [5]:
has_expr = resp[resp.SANGER_MODEL_ID.isin(expr.index)]

for name in ["nutlin", "dabrafenib", "lapatinib", "docetaxel", "paclitaxel"]:
    hits = has_expr[has_expr.DRUG_NAME.str.contains(name, case=False)]
    summary = hits.groupby(["DRUG_NAME", "TARGET"]).size()
    print(summary.to_string() if len(summary) else f"{name}: not found", "\n")

DRUG_NAME      TARGET
Nutlin-3a (-)  MDM2      943 

DRUG_NAME   TARGET
Dabrafenib  BRAF      929 

DRUG_NAME  TARGET     
Lapatinib  EGFR, ERBB2    941 

DRUG_NAME  TARGET                
Docetaxel  Microtubule stabiliser    943 

DRUG_NAME   TARGET                
Paclitaxel  Microtubule stabiliser    933 



In [6]:
DRUGS = ["Nutlin-3a (-)", "Dabrafenib", "Lapatinib", "Docetaxel"]

def make_dataset(drug):
    d = resp[resp.DRUG_NAME == drug].set_index("SANGER_MODEL_ID")
    shared = d.index.intersection(expr.index)
    X = expr.loc[shared]
    y = d.loc[shared, "LN_IC50"]
    tissue = d.loc[shared, "TISSUE"].fillna("UNCLASSIFIED")
    source = models.loc[shared, "data_source"]
    return X, y, tissue, source

for drug in DRUGS:
    X, y, tissue, source = make_dataset(drug)
    out = X.copy()
    out.insert(0, "LN_IC50", y)
    out.insert(1, "TISSUE", tissue)
    out.insert(2, "DATA_SOURCE", source)
    fname = drug.split(" ")[0].lower().replace("-", "")   # e.g. "nutlin3a"
    out.to_csv(f"data/processed/{fname}.csv.gz")
    print(f"{drug:15s} {X.shape}  tissues: {tissue.nunique()}  saved: {fname}.csv.gz")

Nutlin-3a (-)   (943, 5000)  tissues: 32  saved: nutlin3a.csv.gz
Dabrafenib      (929, 5000)  tissues: 32  saved: dabrafenib.csv.gz
Lapatinib       (941, 5000)  tissues: 32  saved: lapatinib.csv.gz
Docetaxel       (943, 5000)  tissues: 32  saved: docetaxel.csv.gz
